# Pipeline convolucional — Detección de anomalías MVTec

## 1. Librerías y configuración del dispositivo

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Usando: {device}")

## 2. Carga de datos y preprocesamiento

In [ ]:
# Parámetros globales
BATCH_SIZE  = 32
NUM_EPOCHS  = 30
IMG_SIZE    = 256

BASE_PATH   = "/kaggle/input/datasets/ipythonx/mvtec-ad"

# 2 texturas + 2 objetos
CATEGORIES = ["screw", "wood", "grid", "hazelnut"]

In [ ]:
# Dataset
class MVTecDataset(Dataset):
    """
    Carga imágenes de MVTec desde una subcarpeta específica.
    Las etiquetas se infieren del nombre del subfolder:
        carpeta 'good'  → label 0  (normal)
        cualquier otra  → label 1  (defecto)
    """
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.samples   = []   # (ruta, label)

        for label_name in sorted(os.listdir(root_dir)):
            label_path = os.path.join(root_dir, label_name)
            if not os.path.isdir(label_path):
                continue
            label = 0 if label_name == "good" else 1
            for fname in sorted(os.listdir(label_path)):
                if fname.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                    self.samples.append((os.path.join(label_path, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

def load_category(category):
    """
    Devuelve train_loader, val_loader y test_loader para una categoría.
    Train y val contienen SOLO imágenes normales (good).
    Test contiene normales + defectuosas para evaluación.
    """
    local_transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5],
                             std=[0.5, 0.5, 0.5])
    ])

    train_full = MVTecDataset(os.path.join(BASE_PATH, category, "train"), local_transform)
    test_ds    = MVTecDataset(os.path.join(BASE_PATH, category, "test"),  local_transform)

    # Split 85/15 sobre imágenes normales de train
    n_total = len(train_full)
    n_val   = int(n_total * 0.15)
    n_train = n_total - n_val
    indices = torch.randperm(n_total).tolist()

    train_ds = Subset(train_full, indices[:n_train])
    val_ds   = Subset(train_full, indices[n_train:])

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

    print(f"[{category}] Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    return train_loader, val_loader, test_loader

## 3. Utilidades compartidas

### 3.1 Early Stopping

In [ ]:
class EarlyStopping:
    def __init__(self, patience=7, verbose=False):
        self.patience   = patience
        self.verbose    = verbose
        self.counter    = 0
        self.best_score = None
        self.early_stop = False
        self.best_loss  = float('inf')

    def __call__(self, val_loss, model):
        if self.best_score is None or val_loss < self.best_loss:
            self.best_loss  = val_loss
            self.best_score = val_loss
            self.counter    = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

### 3.2 Funciones de visualización

In [ ]:
def plot_losses(train_losses, val_losses, category, title_prefix=""):
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses,   label="Val Loss")
    plt.xlabel("Época")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix}Curvas de pérdida — {category}")
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_roc_curve(labels, scores, auc, category, title_prefix=""):
    from sklearn.metrics import roc_curve
    fpr, tpr, _ = roc_curve(labels, scores)
    plt.figure(figsize=(5, 4))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.4f}")
    plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"{title_prefix}Curva ROC — {category}")
    plt.legend()
    plt.tight_layout()
    plt.show()

def show_split(loader, split_name, category, n=5):
    all_images, all_labels = [], []
    for x, y in loader:
        all_images.append(x)
        all_labels.append(y)

    all_images = torch.cat(all_images, dim=0)
    all_labels = torch.cat(all_labels, dim=0)

    n      = min(n, len(all_images))
    idx    = torch.randperm(len(all_images))[:n]
    images = (all_images[idx] * 0.5 + 0.5).clamp(0, 1)
    labels = all_labels[idx]

    fig, axes = plt.subplots(1, n, figsize=(16, 3))
    fig.suptitle(f"[{category}] {split_name}  ({len(all_images)} imágenes)", fontsize=11)
    for i in range(n):
        axes[i].imshow(images[i].permute(1, 2, 0))
        axes[i].set_title("normal" if labels[i] == 0 else "defecto", fontsize=8)
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()

## 4. Autoencoder Convolucional (ConvAE)

### 4.1 Arquitectura

In [ ]:
# Encoder: 3×256×256 → 512×16×16  (4 bloques Conv+BN+ReLU+MaxPool)
# Decoder: 512×16×16 → 3×256×256   (4 bloques Upsample+Conv+BN+ReLU, Tanh final)

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   64,  kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.norm3 = nn.BatchNorm2d(256)
        self.conv4 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.norm4 = nn.BatchNorm2d(512)
        self.maxpool = nn.MaxPool2d(2)
        self.relu    = nn.ReLU()

    def forward(self, x):
        x = self.maxpool(self.relu(self.norm1(self.conv1(x))))   # → 64×128×128
        x = self.maxpool(self.relu(self.norm2(self.conv2(x))))   # → 128×64×64
        x = self.maxpool(self.relu(self.norm3(self.conv3(x))))   # → 256×32×32
        x = self.maxpool(self.relu(self.norm4(self.conv4(x))))   # → 512×16×16
        return x

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.norm1 = nn.BatchNorm2d(256)
        self.conv2 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.norm2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 64,  kernel_size=3, padding=1)
        self.norm3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64,  3,   kernel_size=3, padding=1)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.norm1(self.conv1(self.upsample(x))))  # → 256×32×32
        x = self.relu(self.norm2(self.conv2(self.upsample(x))))  # → 128×64×64
        x = self.relu(self.norm3(self.conv3(self.upsample(x))))  # →  64×128×128
        x = torch.tanh(self.conv4(self.upsample(x)))             # →   3×256×256
        return x

class ConvAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        return self.decoder(self.encoder(x))

### 4.2 Función de entrenamiento

In [ ]:
def train(model, category, train_loader, val_loader, loss_fn=None):

    if loss_fn is None:
        loss_fn = nn.MSELoss()

    optimizer      = optim.Adam(model.parameters(), lr=1e-3)
    scheduler      = torch.optim.lr_scheduler.ReduceLROnPlateau(
                         optimizer, mode='min', factor=0.5, patience=5)
    early_stopping = EarlyStopping(patience=6, verbose=True)

    train_losses, val_losses = [], []

    for epoch in range(NUM_EPOCHS):
        model.train()
        batch_losses = []
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            out  = model(x)
            loss = loss_fn(out, x)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)
        train_losses.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_batch_losses = []
            for x, _ in val_loader:
                x    = x.to(device)
                out  = model(x)
                loss = loss_fn(out, x)
                val_batch_losses.append(loss.item())

        val_loss = np.mean(val_batch_losses)
        val_losses.append(val_loss)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"[{category}] Época {epoch+1:3d} | "
              f"Train: {train_loss:.5f} | Val: {val_loss:.5f} | LR: {current_lr:.6f}")

        scheduler.step(val_loss)
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping en época {epoch+1}")
            break

    return model, train_losses, val_losses

### 4.3 Evaluación AUC-ROC

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluate(model, test_loader, category):
    """
    Score de anomalía: MSE de reconstrucción por imagen.
    Las etiquetas se usan aquí por primera vez para medir AUC-ROC.
    """
    model.eval()
    loss_fn = nn.MSELoss(reduction='none')
    scores, labels_all = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x   = x.to(device)
            out = model(x)
            err = loss_fn(out, x).mean(dim=[1, 2, 3])
            scores.extend(err.cpu().numpy())
            labels_all.extend(y.numpy())

    scores     = np.array(scores)
    labels_all = np.array(labels_all)
    auc        = roc_auc_score(labels_all, scores)
    print(f"[{category}] AUC-ROC: {auc:.4f}")
    return scores, labels_all, auc

### 4.4 Visualización de reconstrucciones

In [ ]:
def plot_reconstructions(model, test_loader, category, n=4):
    """
    Muestra reconstrucciones separadas por etiqueta:
    fila superior → imágenes normales  |  fila inferior → defectuosas
    """
    model.eval()
    normals, defects = [], []
    with torch.no_grad():
        for x, y in test_loader:
            for img, label in zip(x, y):
                if label == 0 and len(normals) < n:
                    normals.append(img)
                if label == 1 and len(defects) < n:
                    defects.append(img)
            if len(normals) >= n and len(defects) >= n:
                break

    fig, axes = plt.subplots(4, n, figsize=(14, 10))
    fig.suptitle(f"Reconstrucciones ConvAE — {category}", fontsize=13)
    row_labels = ["Normal (original)", "Normal (reconstruida)",
                  "Defecto (original)", "Defecto (reconstruida)"]
    for ax, lbl in zip(axes[:, 0], row_labels):
        ax.set_ylabel(lbl, fontsize=8, rotation=90, labelpad=4)

    for i, grupo in enumerate([normals, defects]):
        batch = torch.stack(grupo).to(device)
        with torch.no_grad():
            recon_t = model(batch)
        orig  = (batch.cpu()    * 0.5 + 0.5).clamp(0, 1)
        recon = (recon_t.cpu() * 0.5 + 0.5).clamp(0, 1)

        for j in range(n):
            axes[i*2,   j].imshow(orig[j].permute(1, 2, 0));  axes[i*2,   j].axis('off')
            axes[i*2+1, j].imshow(recon[j].permute(1, 2, 0)); axes[i*2+1, j].axis('off')
            mse = ((orig[j] - recon[j]) ** 2).mean().item()
            axes[i*2+1, j].set_title(f"MSE: {mse:.4f}", fontsize=7)

    plt.tight_layout()
    plt.show()

### 4.5 Ejecución

In [ ]:
# Carga y visualización de imágenes por categoría
loaders_per_category = {}

for category in CATEGORIES:
    print(f"\n{'═'*55}")
    print(f"  CATEGORÍA: {category.upper()}")
    print(f"{'═'*55}")
    train_loader, val_loader, test_loader = load_category(category)
    loaders_per_category[category] = (train_loader, val_loader, test_loader)
    show_split(train_loader, "TRAIN (solo normales)",   category)
    show_split(val_loader,   "VAL   (solo normales)",   category)
    show_split(test_loader,  "TEST  (normal + defecto)", category)

In [ ]:
results_convae = {}

for category in CATEGORIES:
    print(f"\n{'═'*55}")
    print(f"  CATEGORÍA: {category.upper()}")
    print(f"{'═'*55}")

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    train_loader, val_loader, test_loader = loaders_per_category[category]

    model = ConvAE().to(device)
    model, train_losses, val_losses = train(model, category, train_loader, val_loader)

    plot_losses(train_losses, val_losses, category, title_prefix="ConvAE — ")
    plot_reconstructions(model, test_loader, category)

    scores, labels, auc = evaluate(model, test_loader, category)
    plot_roc_curve(labels, scores, auc, category, title_prefix="ConvAE — ")

    results_convae[category] = auc

print(f"\n{'─'*35}")
print("  ConvAE — RESUMEN AUC-ROC")
print(f"{'─'*35}")
for cat, auc in results_convae.items():
    bar = '█' * int(auc * 20)
    print(f"  {cat:<12} {auc:.4f}  {bar}")
print(f"{'─'*35}")
print(f"  Promedio:    {np.mean(list(results_convae.values())):.4f}")

## 5. Autoencoder Variacional Convolucional (Conv VAE)

El Conv VAE extiende el ConvAE añadiendo un espacio latente probabilístico. El encoder no produce un vector determinístico sino una distribución gaussiana (media `mu` y log-varianza `logvar`) sobre el espacio latente. En inferencia se muestrea un vector `z` mediante el reparameterization trick, y el decoder reconstruye la imagen a partir de él. La función de pérdida combina el error de reconstrucción (MSE) con la divergencia KL, que regula la forma del espacio latente y evita que el modelo memorice las imágenes de entrenamiento.

### 5.1 Arquitectura

In [ ]:
# Encoder convolucional compartido con ConvAE (3×256×256 → 512×16×16).
# Las capas fc_mu y fc_logvar proyectan el mapa de características al espacio latente.
# El decoder invierte el proceso: z → 512×16×16 → 3×256×256.

LATENT_DIM = 512   # dimensión del espacio latente
FEAT_SIZE  = 512 * 16 * 16   # 131 072 — tamaño del mapa aplanado

class ConvVAE(nn.Module):
    def __init__(self, latent_dim=LATENT_DIM):
        super().__init__()

        # ── Encoder ──────────────────────────────────────────────────────────
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3,   64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.BatchNorm2d(512), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.fc_mu     = nn.Linear(FEAT_SIZE, latent_dim)
        self.fc_logvar = nn.Linear(FEAT_SIZE, latent_dim)

        # ── Decoder ──────────────────────────────────────────────────────────
        self.fc_decode = nn.Linear(latent_dim, FEAT_SIZE)
        self.decoder_conv = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(128, 64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(64,  3,   3, padding=1), nn.Tanh(),
        )

    def encode(self, x):
        h = self.encoder_conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.decoder_conv(self.fc_decode(z).view(-1, 512, 16, 16))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z          = self.reparameterize(mu, logvar)
        x_hat      = self.decode(z)
        return x_hat, mu, logvar

### 5.2 Función de pérdida

In [ ]:
def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    """
    ELBO negado: MSE de reconstrucción + beta * KL(q(z|x) || p(z)).
    beta=1 corresponde al VAE estándar; valores mayores priorizan
    la regularización del espacio latente (beta-VAE).
    """
    recon = nn.functional.mse_loss(x_hat, x, reduction='mean')
    kl    = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl

### 5.3 Función de entrenamiento

In [ ]:
def train_vae(model, category, train_loader, val_loader, beta=1.0):

    optimizer      = optim.Adam(model.parameters(), lr=1e-3)
    scheduler      = torch.optim.lr_scheduler.ReduceLROnPlateau(
                         optimizer, mode='min', factor=0.5, patience=5)
    early_stopping = EarlyStopping(patience=6, verbose=True)

    train_losses, val_losses = [], []

    for epoch in range(NUM_EPOCHS):
        model.train()
        batch_losses = []
        for x, _ in train_loader:
            x = x.to(device)
            optimizer.zero_grad()
            x_hat, mu, logvar  = model(x)
            loss, recon, kl    = vae_loss(x_hat, x, mu, logvar, beta)
            loss.backward()
            optimizer.step()
            batch_losses.append(loss.item())

        train_loss = np.mean(batch_losses)
        train_losses.append(train_loss)

        model.eval()
        with torch.no_grad():
            val_batch_losses = []
            for x, _ in val_loader:
                x = x.to(device)
                x_hat, mu, logvar = model(x)
                loss, _, _        = vae_loss(x_hat, x, mu, logvar, beta)
                val_batch_losses.append(loss.item())

        val_loss = np.mean(val_batch_losses)
        val_losses.append(val_loss)

        current_lr = optimizer.param_groups[0]['lr']
        print(f"[{category}] Época {epoch+1:3d} | "
              f"Train: {train_loss:.5f} | Val: {val_loss:.5f} | LR: {current_lr:.6f}")

        scheduler.step(val_loss)
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"Early stopping en época {epoch+1}")
            break

    return model, train_losses, val_losses

### 5.4 Evaluación AUC-ROC

In [ ]:
def evaluate_vae(model, test_loader, category):
    """
    Score de anomalía: MSE de reconstrucción por imagen (término recon del ELBO).
    Se excluye el KL porque mide la regularidad del latente, no la fidelidad de la imagen.
    """
    model.eval()
    loss_fn = nn.MSELoss(reduction='none')
    scores, labels_all = [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            x_hat, mu, logvar = model(x)
            err = loss_fn(x_hat, x).mean(dim=[1, 2, 3])
            scores.extend(err.cpu().numpy())
            labels_all.extend(y.numpy())

    scores     = np.array(scores)
    labels_all = np.array(labels_all)
    auc        = roc_auc_score(labels_all, scores)
    print(f"[{category}] AUC-ROC: {auc:.4f}")
    return scores, labels_all, auc

### 5.5 Visualización de reconstrucciones

In [ ]:
def plot_reconstructions_vae(model, test_loader, category, n=4):
    """
    Igual que plot_reconstructions pero extrae x_hat del output del VAE (x_hat, mu, logvar).
    """
    model.eval()
    normals, defects = [], []
    with torch.no_grad():
        for x, y in test_loader:
            for img, label in zip(x, y):
                if label == 0 and len(normals) < n:
                    normals.append(img)
                if label == 1 and len(defects) < n:
                    defects.append(img)
            if len(normals) >= n and len(defects) >= n:
                break

    fig, axes = plt.subplots(4, n, figsize=(14, 10))
    fig.suptitle(f"Reconstrucciones Conv VAE — {category}", fontsize=13)
    row_labels = ["Normal (original)", "Normal (reconstruida)",
                  "Defecto (original)", "Defecto (reconstruida)"]
    for ax, lbl in zip(axes[:, 0], row_labels):
        ax.set_ylabel(lbl, fontsize=8, rotation=90, labelpad=4)

    for i, grupo in enumerate([normals, defects]):
        batch = torch.stack(grupo).to(device)
        with torch.no_grad():
            x_hat, _, _ = model(batch)
        orig  = (batch.cpu()  * 0.5 + 0.5).clamp(0, 1)
        recon = (x_hat.cpu() * 0.5 + 0.5).clamp(0, 1)

        for j in range(n):
            axes[i*2,   j].imshow(orig[j].permute(1, 2, 0));  axes[i*2,   j].axis('off')
            axes[i*2+1, j].imshow(recon[j].permute(1, 2, 0)); axes[i*2+1, j].axis('off')
            mse = ((orig[j] - recon[j]) ** 2).mean().item()
            axes[i*2+1, j].set_title(f"MSE: {mse:.4f}", fontsize=7)

    plt.tight_layout()
    plt.show()

### 5.6 Ejecución

In [ ]:
results_vae = {}

for category in CATEGORIES:
    print(f"\n{'═'*55}")
    print(f"  CATEGORÍA: {category.upper()}")
    print(f"{'═'*55}")

    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    train_loader, val_loader, test_loader = loaders_per_category[category]

    model = ConvVAE().to(device)
    model, train_losses, val_losses = train_vae(model, category, train_loader, val_loader)

    plot_losses(train_losses, val_losses, category, title_prefix="Conv VAE — ")
    plot_reconstructions_vae(model, test_loader, category)

    scores, labels, auc = evaluate_vae(model, test_loader, category)
    plot_roc_curve(labels, scores, auc, category, title_prefix="Conv VAE — ")

    results_vae[category] = auc

print(f"\n{'─'*35}")
print("  Conv VAE — RESUMEN AUC-ROC")
print(f"{'─'*35}")
for cat, auc in results_vae.items():
    bar = '█' * int(auc * 20)
    print(f"  {cat:<12} {auc:.4f}  {bar}")
print(f"{'─'*35}")
print(f"  Promedio:    {np.mean(list(results_vae.values())):.4f}")

## 6. Conclusiones

*(Por completar tras la ejecución.)*

Este notebook compara dos variantes de autoencoder convolucional para detección de anomalías en MVTec:

- **ConvAE** — encoder/decoder convolucional determinístico con pérdida MSE. Preserva la estructura espacial de la imagen, lo que debería mejorar la localización de defectos respecto al baseline FC.
- **Conv VAE** — mismo backbone convolucional pero con espacio latente gaussiano y pérdida ELBO (MSE + KL). La regularización del latente penaliza representaciones demasiado específicas, forzando al modelo a capturar únicamente la estructura normal generalizable.

La comparación central es si el VAE aporta una mejora real en AUC-ROC respecto al ConvAE, o si el coste de la regularización (pérdida de fidelidad de reconstrucción) no compensa en este dominio.